# 🏗️ Notebook 1 — Code Deployment (CI/CD): Requirements, Architecture & a Pipeline Runner

Welcome! In this lab we design a **code deployment system** — the machinery that turns a Git commit into running code in production. Think **GitHub Actions**, **GitLab CI**, **Jenkins**, **ArgoCD**, **Spinnaker**.

By the end of this notebook you will be able to:

1. Explain *what* a CI/CD system has to do and *why*.
2. Sketch a high-level architecture (trigger → build → test → artifact → deploy → verify → rollback).
3. **Run** a tiny pipeline executor in pure Python that walks from a **bad** sequential script to a **good** DAG runner with parallel stages.

> 🎯 **Audience:** complete beginners in system design. No Docker, no Kubernetes, no cloud required — everything runs in this notebook.

## 🛠️ Setup

```bash
cd 06-system-designs/code-deployment
uv sync
```

Then in VS Code pick the `.venv` kernel (top-right). If it doesn't appear:
`Cmd+Shift+P` → **Reload Window**.

## 1. What are we actually designing?

Every time an engineer runs `git push`, a *lot* has to happen before users see the change:

```
commit ──▶ build ──▶ test ──▶ package ──▶ deploy ──▶ verify ──▶ (rollback?)
```

A **CI/CD system** owns that whole pipe. Its job is to make the path from *code written* to *code running in production* **safe, fast, and boring**.

### Vocabulary (plain English)

| Term | What it means |
|---|---|
| **CI — Continuous Integration** | Every push is built & tested automatically. |
| **CD — Continuous Delivery** | Every green build is *ready* to ship (a human clicks "go"). |
| **CD — Continuous Deployment** | Every green build ships *automatically*. |
| **Pipeline** | The recipe: ordered steps from commit to prod. |
| **Stage / Job** | One step of the pipeline (e.g. `unit-tests`). |
| **Artifact** | The thing you built (a container image, a `.jar`, a `.zip`). Should be **immutable**. |
| **Deployment** | The act of putting an artifact into an environment (dev/staging/prod). |
| **Rollback** | Putting the *previous* artifact back, fast, when the new one misbehaves. |

If you remember one thing: **we ship *artifacts*, not commits.** The commit is just the recipe; the artifact is the cake.

## 2. Requirements

### Functional
- Triggered by **push**, **pull request**, **schedule**, or **manual button**.
- Run a **DAG** of stages (some can run in parallel).
- Produce an **immutable artifact** with a unique ID (e.g. an image digest).
- Deploy to multiple environments: `dev`, `staging`, `prod`.
- Support deploy strategies: **rolling**, **blue/green**, **canary**.
- **Automatic rollback** if post-deploy health checks fail.
- Show logs + status to engineers in near real time.

### Non-functional
- **Reproducible builds** — same commit, same artifact, every time.
- End-to-end time for a small service: **under 15 minutes** is a common target.
- **Never** deploy a red build silently.
- Multi-tenant, isolated: one team's build must not leak secrets to another team.
- Auditable: who deployed what, where, when.

## 3. Back-of-envelope scale

Let's pretend we're running CI for a mid-size company:

- 10,000 engineers × 5 pushes/day ≈ **50,000 builds/day** (~0.6/sec average).
- Peak is usually 5–10× average → **~6 builds/second** at 10am.
- Average build uses 6 CPU / 8 GB RAM for ~15 min on an ephemeral worker.
- Artifacts: ~200 MB each × 50k/day ≈ **10 TB/day** of new images (before dedup).

Takeaways:
1. The **worker pool** is the expensive thing — design for **elasticity**.
2. The **artifact store** must support cheap **content-addressed** storage (the same layer stored once).
3. The **scheduler / queue** must handle bursty, parallelizable workloads.

## 4. High-level architecture

```
           ┌──────────┐  webhook   ┌──────────────┐
  Git  ───▶│ Trigger  │──────────▶│ Pipeline Svc │
           └──────────┘            └──────┬───────┘
                                          │ enqueue run
                                          ▼
                                   ┌──────────────┐       ┌──────────────┐
                                   │  Scheduler   │──────▶│ Worker Pool  │  (ephemeral)
                                   └──────┬───────┘       └──────┬───────┘
                                          │ status/logs          │ push image
                                          ▼                      ▼
                                   ┌──────────────┐       ┌──────────────┐
                                   │   Metadata   │       │   Artifact   │
                                   │      DB      │       │   Registry   │ (content-addressed)
                                   └──────┬───────┘       └──────┬───────┘
                                          │                      │
                                          ▼                      ▼
                                   ┌──────────────────────────────────┐
                                   │   Deployer (rolling / canary)    │──▶ K8s / target envs
                                   └──────┬───────────────────────────┘
                                          │ watches
                                          ▼
                                   ┌──────────────┐    breach    ┌──────────────┐
                                   │  SLO/Metrics │─────────────▶│ Auto-rollback│
                                   └──────────────┘              └──────────────┘
```

Why split it like this?

- **One responsibility per box.** Easier to scale and reason about.
- **Stateless services** (trigger, pipeline, deployer) scale horizontally.
- **Stateful pieces** (metadata DB, artifact registry) are picked for their access pattern: the DB is *transactional* (who deployed what), the registry is *content-addressed blob storage*.
- **The deployer is separate from the builder** — separating "make the artifact" from "install the artifact" is the single most important design choice in CI/CD.

## 5. A tiny pipeline runner — bad → better → best

Enough theory. Let's write a pipeline runner and improve it three times.

### 🔴 Attempt 1 (BAD): a straight-line shell-style script

Everything is sequential and hard-coded. No parallelism, no retries, no failure handling — if `unit_test` fails we still go on to deploy. 😬

In [1]:
# 🔴 BAD: sequential, fire-and-forget
def bad_pipeline():
    print("build   ... ok")
    print("unit    ... FAIL")   # pretend this failed
    print("integ   ... ok")
    print("deploy  ... ok")     # we deployed anyway 😱

bad_pipeline()

build   ... ok
unit    ... FAIL
integ   ... ok
deploy  ... ok


**Problems**

1. Failure in one stage does not stop later stages.
2. No way to say *"integ depends on build"* — it's just line order.
3. Independent stages (`unit` and `integ`) run sequentially → slow.
4. No structured result we can store in a DB or show in a UI.

### 🟡 Attempt 2 (BETTER): typed stages with dependencies + stop-on-failure

In [2]:
# 🟡 BETTER: typed stages, topo-sorted, stop on failure
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class Stage:
    name: str
    run: Callable[[], bool]          # returns True = success
    depends_on: list[str] = field(default_factory=list)

def topo_order(stages: list[Stage]) -> list[Stage]:
    """Return stages in an order that respects depends_on (Kahn's algorithm)."""
    by_name = {s.name: s for s in stages}
    indeg = {s.name: 0 for s in stages}
    children: dict[str, list[str]] = {s.name: [] for s in stages}
    for s in stages:
        for dep in s.depends_on:
            indeg[s.name] += 1
            children[dep].append(s.name)
    ready = [n for n, d in indeg.items() if d == 0]
    order = []
    while ready:
        n = ready.pop(0)
        order.append(by_name[n])
        for c in children[n]:
            indeg[c] -= 1
            if indeg[c] == 0:
                ready.append(c)
    if len(order) != len(stages):
        raise ValueError("cycle in pipeline DAG")
    return order

def run_pipeline_v2(stages: list[Stage]) -> bool:
    for s in topo_order(stages):
        print(f"▶ {s.name:<8} ", end="")
        ok = s.run()
        print("✅" if ok else "❌")
        if not ok:
            print(f"✋ stopping: {s.name} failed")
            return False
    return True

pipeline = [
    Stage("build",  lambda: True),
    Stage("unit",   lambda: False, depends_on=["build"]),   # fails
    Stage("integ",  lambda: True,  depends_on=["build"]),
    Stage("deploy", lambda: True,  depends_on=["unit", "integ"]),
]
print("pipeline result:", run_pipeline_v2(pipeline))

▶ build    ✅
▶ unit     ❌
✋ stopping: unit failed
pipeline result: False


**Better!** We now:

- Model the pipeline as a **DAG** — dependencies are explicit.
- **Stop** on first failure — no more accidental deploys on red tests.
- Have a list of typed `Stage` objects we could serialise to JSON / store in a DB.

**Still missing:** `unit` and `integ` both depend only on `build`. They could run *in parallel*. Today we still run them one after another.

### 🟢 Attempt 3 (BEST): parallel execution of independent stages

We run all stages whose dependencies are *already done* at the same time using a thread pool. This is how real CI systems (GitHub Actions `needs:`, GitLab `needs:`, Tekton, Argo Workflows) work.

In [3]:
# 🟢 BEST: level-by-level parallel DAG execution
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def run_pipeline_parallel(stages: list[Stage], max_workers: int = 4) -> dict:
    remaining = {s.name: s for s in stages}
    done: dict[str, bool] = {}
    results: dict[str, dict] = {}
    aborted = False

    def ready() -> list[Stage]:
        return [s for s in remaining.values()
                if all(d in done and done[d] for d in s.depends_on)]

    start_all = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        while remaining and not aborted:
            batch = ready()
            if not batch:
                # Nothing runnable ⇒ blocked by an earlier failure.
                break
            futures = {pool.submit(s.run): s for s in batch}
            for s in batch:
                remaining.pop(s.name)
            for fut in as_completed(futures):
                s = futures[fut]
                t0 = time.time()
                ok = fut.result()
                results[s.name] = {"ok": ok, "duration": round(time.time() - t0, 3)}
                done[s.name] = ok
                print(f"  {s.name:<8} {'✅' if ok else '❌'}")
            if any(not v for v in done.values()):
                print("✋ aborting remaining stages (earlier failure)")
                aborted = True

    results["_total_seconds"] = round(time.time() - start_all, 3)
    return results

def slow(ok=True, secs=0.4):
    """Pretend a stage does real work."""
    def _run():
        time.sleep(secs)
        return ok
    return _run

pipeline = [
    Stage("build",  slow(True, 0.5)),
    Stage("unit",   slow(True, 0.8), depends_on=["build"]),
    Stage("integ",  slow(True, 0.8), depends_on=["build"]),
    Stage("lint",   slow(True, 0.3), depends_on=["build"]),
    Stage("deploy", slow(True, 0.4), depends_on=["unit", "integ", "lint"]),
]
res = run_pipeline_parallel(pipeline, max_workers=3)
print("\nresults:", res)
print(f"\nWall-clock: {res['_total_seconds']}s  (sequential would be ~2.8s)")

  build    ✅


  lint     ✅


  integ    ✅
  unit     ✅


  deploy   ✅

results: {'build': {'ok': True, 'duration': 0.0}, 'lint': {'ok': True, 'duration': 0.0}, 'integ': {'ok': True, 'duration': 0.0}, 'unit': {'ok': True, 'duration': 0.0}, 'deploy': {'ok': True, 'duration': 0.0}, '_total_seconds': 2.146}

Wall-clock: 2.146s  (sequential would be ~2.8s)


**What changed**

- `unit`, `integ`, `lint` run **concurrently** — total time is roughly the *longest path* (~1.7s), not the sum of all stages (~2.8s).
- Each level is computed from the dependency graph, not hard-coded. Add a new stage and the scheduler "just works".
- Failures still stop the pipeline from doing the *next* level.

> This is the core scheduling loop inside every modern CI system. The rest (distributed workers, caching, log streaming, UI) is plumbing around this idea.

## 6. Where we'll go next

- **Notebook 2** → data model (Pipeline, Run, Stage, Artifact, Deployment) + HTTP APIs + why artifacts must be immutable.
- **Notebook 3** → deploy strategies (rolling, blue/green, canary) with a runnable **SLO-gated auto-rollback**.

### 🔑 Key takeaways from this notebook

1. CI/CD = a DAG of stages turning a commit into a running artifact.
2. Model it as *data* (stages + edges), not as a script.
3. Parallelize independent stages; stop the whole run on first failure.
4. Separate **building the artifact** from **deploying** it — the single most important architectural rule.